# 실습 14 · 분자 도킹 (AutoDock Vina & DiffDock)
### 약물이 표적 단백질에 어떻게·얼마나 세게 붙는지 예측

**왜 중요한가 (수강생 최다 요청 주제)**
**도킹(docking)** 은 약물 분자가 표적 단백질의 결합부위에 **어떤 자세로 얼마나 강하게**
결합하는지를 컴퓨터로 예측합니다. 구조 기반 신약설계(SBDD)의 핵심 기술입니다.
- **전통적 도킹**: AutoDock Vina (물리/스코어 함수 기반) — 현장 표준
- **딥러닝 도킹**: DiffDock (생성모델 기반) — 최신, 빠르고 별도 pocket 지정 불필요

**이 노트북의 흐름**
1. 도킹 개념 (리셉터/리간드/결합에너지)
2. **AutoDock Vina** 로 실제 도킹 실행 (BACE-1 표적 + 리간드)
3. 결합에너지 해석 & 결합 자세 3D 시각화
4. **DiffDock**(딥러닝 도킹) 소개와 사용법

> GPU 권장. 설치에 수 분 걸릴 수 있습니다.


## 1. 도킹의 기본 개념
- **리셉터(receptor)**: 표적 단백질 (예: BACE-1, 알츠하이머 효소)
- **리간드(ligand)**: 후보 약물 분자
- **결합에너지(binding affinity, kcal/mol)**: **음수로 클수록**(예: -9) 강한 결합
- **결합 자세(pose)**: 리간드가 pocket 에 놓이는 3D 배치

도킹 프로그램은 수많은 자세를 탐색해 **에너지가 가장 낮은(=안정) 자세**를 찾습니다.


In [ ]:
# 도킹 도구 설치: AutoDock Vina(파이썬), RDKit, Meeko(리간드 준비)
!pip install vina meeko rdkit py3Dmol requests -q 2>/dev/null
print("설치 시도 완료 (버전에 따라 시간이 걸릴 수 있음)")
import requests, py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem

## 2. 표적 단백질 & 리간드 준비
실제 알츠하이머 표적 **BACE-1**(PDB: 4B05) 구조를 받고,
후보 리간드를 SMILES 에서 3D 로 생성합니다.


In [ ]:
# 표적 단백질 다운로드
r = requests.get("https://files.rcsb.org/download/4B05.pdb", timeout=60)
open("receptor_raw.pdb","w").write(r.text)
print("BACE-1 표적 구조 다운로드 완료")

# 리간드: BACE 저해제 후보 하나 (SMILES → 3D 구조 → 에너지 최소화)
ligand_smiles = "Fc1cc(cc(F)c1)CC(NC(=O)C)C(O)C[NH2+]Cc1cc(ccc1)C(C)(C)C"
mol = Chem.MolFromSmiles(ligand_smiles)
mol = Chem.AddHs(mol)                       # 수소 추가
AllChem.EmbedMolecule(mol, randomSeed=42)   # 3D 좌표 생성
AllChem.MMFFOptimizeMolecule(mol)           # 에너지 최소화
Chem.MolToMolFile(mol, "ligand.mol")
print("리간드 3D 구조 생성 완료 (원자 수:", mol.GetNumAtoms(), ")")

## 3. ⭐ 도킹 실행 개념 & 코드
실제 Vina 도킹은 아래 단계를 거칩니다. (환경/버전 의존성이 커서 핵심 흐름 위주로 제시)

**단계**
1. 리셉터·리간드를 PDBQT 형식으로 변환 (전하·원자타입 부여)
2. 결합부위(search box) 중심좌표·크기 지정
3. Vina 실행 → 여러 pose 와 각 결합에너지 산출

아래는 Vina 파이썬 API 로 도킹을 수행하는 **실전 코드 골격**입니다.


In [ ]:
# ▼ AutoDock Vina 실전 코드 (리셉터/리간드 PDBQT 준비 후 실행)
# from vina import Vina
# v = Vina(sf_name='vina')
# v.set_receptor('receptor.pdbqt')           # 준비된 리셉터
# v.set_ligand_from_file('ligand.pdbqt')     # 준비된 리간드
# v.compute_vina_maps(center=[x,y,z], box_size=[20,20,20])  # 결합부위 상자
# v.dock(exhaustiveness=8, n_poses=10)       # 도킹 실행
# v.write_poses('out.pdbqt', n_poses=5, overwrite=True)
# energies = v.energies()                     # 각 pose 의 결합에너지
# print('최적 결합에너지:', energies[0][0], 'kcal/mol')

print("※ PDBQT 준비는 MGLTools/Meeko 로 합니다.")
print("  개념 이해가 목적이면 다음 셀의 '미리 계산된 결과'로 해석 연습을 하세요.")

In [ ]:
# 도킹 결과 해석 연습 — 여러 후보의 결합에너지 비교 (예시 값)
import pandas as pd, matplotlib.pyplot as plt
docking_results = pd.DataFrame({
    "후보물질": ["Ligand-A","Ligand-B","Ligand-C","Ligand-D","알려진 약물"],
    "결합에너지(kcal/mol)": [-9.8, -8.2, -7.1, -6.5, -10.3],
})
docking_results = docking_results.sort_values("결합에너지(kcal/mol)")
plt.figure(figsize=(7,3.5))
colors = ["#C44E52" if e<-9 else "#4C72B0" for e in docking_results["결합에너지(kcal/mol)"]]
plt.barh(docking_results["후보물질"], docking_results["결합에너지(kcal/mol)"], color=colors)
plt.xlabel("결합에너지 (음수로 클수록 강한 결합 ←)")
plt.title("도킹 기반 후보물질 순위"); plt.tight_layout(); plt.show()
print("→ -9 kcal/mol 이하면 유망(빨강). 알려진 약물과 비교해 우선순위 결정")

## 4. 결합 자세(pose) 3D 시각화
단백질 pocket 안에 리간드가 놓인 모습을 보는 것은 도킹 해석의 핵심입니다.
표적 단백질과 리간드를 함께 렌더링합니다.


In [ ]:
# 리셉터(리본) + 리간드(막대) 함께 표시 개념 시연
view = py3Dmol.view(width=680, height=480)
view.addModel(open("receptor_raw.pdb").read(), "pdb")
view.setStyle({"cartoon": {"color": "lightgray"}})          # 단백질은 회색 리본
# 결합부위 근처 잔기를 막대로 강조하면 pocket 이 보임
view.addStyle({"resn": ["ASP","THR","TYR"]}, {"stick": {"colorscheme":"orangeCarbon"}})
view.zoomTo(); view.show()
print("실제 도킹에선 여기에 리간드 pose 를 겹쳐 상호작용(수소결합 등)을 분석")

## 5. ⭐ DiffDock — 딥러닝 기반 도킹 (최신)
**DiffDock**(MIT, 2023)은 확산모델(diffusion)로 결합 자세를 **생성**합니다.
- 전통 도킹과 달리 **결합부위를 미리 지정할 필요 없음**(blind docking)
- **빠르고** 여러 그럴듯한 자세를 제시
- 수강생 피드백의 "도킹을 위한 딥러닝"에 정확히 해당하는 최신 기술

**사용법**
- GitHub `gcorso/DiffDock` 의 Colab 데모 또는 HuggingFace Space 사용
- 입력: 단백질 PDB + 리간드 SMILES → 출력: 예측 결합 자세들
```python
# 개념 예시 (공식 저장소 데모 기준)
# !git clone https://github.com/gcorso/DiffDock.git
# python -m inference --protein_path protein.pdb --ligand "SMILES" --out_dir results/
```

| | AutoDock Vina | DiffDock (딥러닝) |
|---|---|---|
| 방식 | 물리 스코어 함수 | 생성형 확산모델 |
| pocket 지정 | 필요 | 불필요(blind) |
| 속도 | 보통 | 빠름 |
| 강점 | 검증된 표준, 에너지값 | 미지 pocket, 유연성 |


## 정리 & 현장 응용
- 도킹 = 표적에 대한 **결합 자세 + 결합에너지** 예측 (구조 기반 신약설계 핵심)
- **AutoDock Vina**: 검증된 표준 (리셉터/리간드 PDBQT → box 지정 → 도킹)
- **DiffDock**: 딥러닝 기반, blind docking, 최신 워크플로우
- 전체 파이프라인: 예제13(구조) → 14(도킹) → 활성 후보 선별 → 실험 검증
- 실무 팁: 도킹 점수는 **상대 비교·1차 선별**용, 최종은 실험으로 확인
